In [1]:
# Importe le module sys pour connaître l'interpréteur Python actuellement utilisé.
import sys
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
# Importe Path pour afficher le dossier courant du notebook.
from pathlib import Path

# Importe boto3, le SDK Python permettant de communiquer avec AWS.
import boto3

import pandas as pd
import pyarrow.ipc as ipc
import pyarrow.parquet as pq
import pyarrow as pa

# Importe le module de systèmes de fichiers de PyArrow.
from pyarrow import fs


#Affiche le chemin de l'interpréteur Python utilisé par le notebook AWS.
print("Python :", sys.executable)

In [2]:


s3 = fs.S3FileSystem(region="us-east-1")


# Indique le nom exact du bucket S3 contenant les données.
# Remplace cette valeur par le nom réel de ton bucket.
BUCKET_NAME = "mushroom-ml-mikael-2026-000"

# Équivalent AWS de PROJECT_ROOT.
# Dans notre cas, les fichiers Arrow sont dans le préfixe raw/.
PROJECT_ROOT = f"{BUCKET_NAME}/raw"


# Demande à PyArrow de regarder directement dans raw/,
# sans parcourir récursivement d'éventuels sous-dossiers.
selector = fs.FileSelector(
    PROJECT_ROOT,
    recursive=False
)

# Récupère la liste des fichiers présents dans raw/.
files_information = s3.get_file_info(selector)


# Conserve uniquement les vrais fichiers dont le nom finit par .arrow.
arrow_files = sorted(
    file_information.path
    for file_information in files_information
    if file_information.type == fs.FileType.File
    and file_information.path.endswith(".arrow")
)


# Parcourt les fichiers Arrow trouvés.
for path in arrow_files:

    # Affiche seulement le nom du fichier, sans le bucket ni raw/.
    print(Path(path).name)


# Définit l'emplacement où nous écrirons les fichiers Parquet.
PARQUET_DIR = f"{BUCKET_NAME}/processed/v1"


# Affiche la future destination.
print("Destination Parquet :", f"s3://{PARQUET_DIR}")

In [3]:

merged_table = None

for arrow_file in arrow_files:

    with s3.open_input_stream(arrow_file) as source:
        table = ipc.open_stream(source).read_all()

    file_name = Path(arrow_file).name

    if "train" in file_name:
        split = "train"
    elif "validation" in file_name:
        split = "validation"
    else:
        split = "test"

#    table = table.append_column(
#        "split",
#        pa.array([split] * table.num_rows)
#    )

    #Adding columns telling if the entry is generated or not (natural / generated)

    origin = pa.array(
        ["natural"] * table.num_rows
    )
    
    table = table.append_column(
        "origin",
        origin
    )

    if merged_table is None:
        merged_table = table
    else:
        merged_table = pa.concat_tables([merged_table, table])

    print(split, ":", table.num_rows, "lignes")
    print("Total :", merged_table.num_rows, "lignes")

print("Lignes totales :", merged_table.num_rows)
print("Colonnes :", merged_table.column_names)


In [4]:
PARQUET_DIR = f"{PROJECT_ROOT}/parquet"

PARQUET_PATH = PARQUET_DIR + "/mushrooms_merged.parquet"

pq.write_table(
    merged_table,
    PARQUET_PATH,
    filesystem=s3,
    compression="zstd"
)

